# 2주차 6교시 · 전처리 기준을 배우고 파일로 저장해 다시 사용하기

**학생용 지적실험노트**

오늘의 질문: **오늘 계산한 중앙값·평균·범주 목록을 다음 실행에서도 똑같이 사용할 수 있을까요?**

오늘의 완성 결과: train에서 배운 기준으로 36열 표를 만들고 전처리기를 matchmaking_preprocessor.joblib로 저장합니다.

## 시작하기 전에 폴더를 확인하세요

- `activity_batches`는 강사가 ZIP 안에 미리 넣어 준 폴더입니다. 학생이 새로 만들지 않습니다.
- ZIP을 먼저 압축 해제하고, 압축을 풀어서 생긴 최상위 폴더 전체를 VS Code에서 엽니다.
- 이 노트북이나 CSV 파일을 다른 폴더로 따로 옮기지 않습니다.
- 자세한 순서는 `week2/00_학생용_먼저읽기_압축해제와VSCode열기.md`를 확인합니다.

각 Q에서 ① 실행 전 예상 ② ___ 채우기 ③ 실행 ④ 값 하나 바꾸기 ⑤ 결과 이유 한 문장을 지킵니다. 오류 메시지에 ___가 보이면 고장이 아니라 아직 코드 빈칸이 남았다는 뜻입니다.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

def find_week2_data_dir():
    """노트북을 어느 교시 폴더에서 열어도 2주차 데이터 폴더를 찾습니다."""
    here = Path.cwd().resolve()
    for base in [here, *here.parents]:
        candidates = [
            base / "course_materials" / "matchmaking_data" / "02_week2",
            base / "matchmaking_data" / "02_week2",
            base / "02_week2",
        ]
        for candidate in candidates:
            if (candidate / "activity_batches").exists():
                return candidate
    raise FileNotFoundError("02_week2 데이터 폴더를 찾지 못했습니다. python_lecture 프로젝트 안에서 노트북을 열어 주세요.")

DATA_DIR = find_week2_data_dir()
WEEK2_DIR = DATA_DIR.parents[1] / "week2"
ARTIFACT_DIR = WEEK2_DIR / "artifacts"
ARTIFACT_DIR.mkdir(exist_ok=True)
print("데이터 폴더:", DATA_DIR)
batch_files = sorted((DATA_DIR / "activity_batches").glob("activity_batch_*.csv"))
frames = []
for file_path in batch_files:
    one_batch = pd.read_csv(file_path)
    one_batch["source_file"] = file_path.name
    frames.append(one_batch)
activity = pd.concat(frames, ignore_index=True)

activity["activity_dt"] = pd.to_datetime(activity["activity_date"], errors="coerce")
activity["stay_time_min_num"] = pd.to_numeric(activity["stay_time_min"], errors="coerce")
activity["active_dt"] = activity["activity_dt"].where(activity["logged_in"])

member_activity = activity.groupby("member_id").agg(
    active_days=("logged_in", "sum"),
    total_app_opens=("app_opens", "sum"),
    total_stay_time_min=("stay_time_min_num", "sum"),
    total_profile_views=("profile_views", "sum"),
    total_recommendations_viewed=("recommendations_viewed", "sum"),
    total_messages_sent=("messages_sent", "sum"),
    avg_reply_delay_sec=("avg_reply_delay_sec", "mean"),
    last_active_date=("active_dt", "max"),
).reset_index()
reference_date = pd.Timestamp("2026-07-01")
member_activity["days_since_last_activity"] = (
    reference_date - member_activity["last_active_date"]
).dt.days

members = pd.read_csv(DATA_DIR / "members.csv")
member_columns = [
    "member_id", "gender", "age", "residence_region",
    "membership_tier", "signup_channel", "meeting_scheduled"
]
model_table = members[member_columns].merge(
    member_activity, on="member_id", how="left", validate="one_to_one"
)

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

numeric_features = [
    "age", "active_days", "total_app_opens", "total_stay_time_min",
    "total_profile_views", "total_recommendations_viewed",
    "total_messages_sent", "avg_reply_delay_sec", "days_since_last_activity"
]
categorical_features = [
    "gender", "residence_region", "membership_tier", "signup_channel"
]
X = model_table[numeric_features + categorical_features]
y = model_table["meeting_scheduled"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
numeric_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])
categorical_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])
preprocessor = ColumnTransformer([
    ("num", numeric_pipe, numeric_features),
    ("cat", categorical_pipe, categorical_features),
], verbose_feature_names_out=False).set_output(transform="pandas")
import joblib


## Q1. train 800명으로 빈값 대체값과 변환 기준을 배웁니다

**먼저 예상하기**

- fit_transform을 실행하면 전처리기 안에는 무엇이 저장될까요?
- 내 예상: ____________________

**왜 이 일을 하나요?** test 정보를 미리 보지 않고 운영에서 재사용할 기준을 만듭니다.

**현재 자료구조** train_processed는 800행×36열 DataFrame입니다.

**코드 읽기 도움** fit_transform은 train에서 기준을 배우는 fit과 그 기준을 적용하는 transform을 연달아 합니다.

**정상 결과** (800, 36)과 pandas DataFrame

**오류가 나면** NotFittedError 또는 출력이 ndarray. fit_transform을 train에 사용하고 set_output 설정을 확인합니다.


In [ ]:
# train 데이터에서 전처리 기준을 배우면서 동시에 변환합니다.
___ = ___.___(___)
# 변환된 train 표가 몇 행·몇 열인지 확인합니다.
print(train_processed.shape)
# 변환 결과가 DataFrame인지 확인합니다.
print(type(train_processed))

## Q2. test 200명에는 train에서 배운 기준만 적용합니다

**먼저 예상하기**

- test에서 중앙값과 범주 목록을 다시 계산하면 왜 공정한 확인이 아닐까요?
- 내 예상: ____________________

**왜 이 일을 하나요?** 시험지에서 새 기준을 배우는 데이터 누수를 막습니다.

**현재 자료구조** test_processed는 200행×36열 DataFrame이며 train_processed와 열이 같습니다.

**코드 읽기 도움** columns.equals는 두 DataFrame의 열 이름과 순서가 같은지 확인합니다.

**정상 결과** (200, 36)과 True

**오류가 나면** 열 수 불일치 또는 test에 fit 사용. 코드 전체에서 fit이라는 글자가 train 줄에만 있는지 찾습니다.


In [ ]:
# train에서 이미 배운 전처리기를 test 데이터에 적용합니다.
___ = preprocessor.___(___)
# 변환된 test 표가 몇 행·몇 열인지 확인합니다.
print(___.shape)
# train과 test의 열 이름과 순서가 같은지 확인합니다.
print(train_processed.columns.equals(___.columns))

## Q3. 변환한 두 표의 크기와 남은 빈값을 검사합니다

**먼저 예상하기**

- 결측값이 0개라는 사실만으로 전처리가 끝났다고 말할 수 있을까요?
- 내 예상: ____________________

**왜 이 일을 하나요?** 전처리 완료를 눈대중이 아니라 숫자로 확인합니다.

**현재 자료구조** missing_train과 missing_test는 정수이며 둘 다 0이어야 합니다.

**코드 읽기 도움** 첫 sum은 열별 결측 수, 두 번째 sum은 그 수를 모두 더한 전체 결측 수입니다.

**정상 결과** 0 0

**오류가 나면** AssertionError. 숫자/범주 Pipeline 결과를 나누어 확인합니다.


In [ ]:
# train 변환 결과에 남은 결측치 개수를 모두 셉니다.
___ = ___.___().sum().sum()
# test 변환 결과에 남은 결측치 개수를 모두 셉니다.
___ = ___.___().sum().sum()
# train 결과의 행·열 크기가 예상과 같은지 확인합니다.
assert ___.shape == (800, 36)
# test 결과의 행·열 크기가 예상과 같은지 확인합니다.
assert ___.shape == (200, 36)
# 두 표의 전체 결측치 개수를 출력합니다.
print(___, ___)

## Q4. 학습이 끝난 전처리기를 joblib 파일로 저장합니다

**먼저 예상하기**

- Python을 종료해도 전처리 기준을 남기려면 무엇을 저장해야 할까요?
- 내 예상: ____________________

**왜 이 일을 하나요?** 서버나 다음 날 실행에서 같은 기준을 다시 사용합니다.

**현재 자료구조** artifact_path는 Path, 저장 파일 안에는 학습이 끝난 preprocessor Python 객체가 직렬화됩니다.

**코드 읽기 도움** dump는 메모리에 있는 객체를 파일로 저장하고, stat().st_size는 파일 크기를 바이트로 확인합니다.

**정상 결과** True와 0보다 큰 파일 크기

**오류가 나면** PermissionError 또는 파일 크기 0. artifacts 폴더가 있는지와 파일을 다른 프로그램이 잡고 있지 않은지 확인합니다.


In [ ]:
# 저장할 joblib 파일의 위치를 만듭니다.
artifact_path = ARTIFACT_DIR / "matchmaking_preprocessor.___"
# 학습이 끝난 전처리기 객체를 파일로 저장합니다.
joblib.___(___, artifact_path)
# 파일이 실제로 생겼는지와 파일 크기를 확인합니다.
print(___.exists(), artifact_path.stat().st_size)

## Q5. 저장한 전처리기를 다시 불러와 같은 결과인지 비교합니다

**먼저 예상하기**

- 파일이 존재한다는 확인만으로 재사용할 수 있다고 말할 수 있을까요?
- 내 예상: ____________________

**왜 이 일을 하나요?** 파일이 존재하는 것뿐 아니라 실제로 재사용 가능한지 검증합니다.

**현재 자료구조** loaded_preprocessor는 새로 불러온 객체, test_again은 200행×36열 DataFrame입니다.

**코드 읽기 도움** allclose는 부동소수점의 아주 작은 표현 차이를 허용하면서 두 숫자 배열이 같은지 비교합니다.

**정상 결과** (200, 36) True

**오류가 나면** FileNotFoundError 또는 False. 저장 경로와 load 경로가 같은지, 저장 전에 fit을 했는지 확인합니다.


In [ ]:
# 저장한 전처리기 객체를 파일에서 다시 불러옵니다.
loaded_preprocessor = ___.load(___)
# 불러온 전처리기를 test 데이터에 다시 적용합니다.
test_again = loaded_preprocessor.___(___)
# 기존 test 결과와 다시 만든 결과가 같은지 비교합니다.
same = np.allclose(test_processed.to_numpy(), test_again.to_numpy())
# 두 결과가 같다는 조건을 검사합니다.
assert same
# 다시 만든 결과의 크기와 비교 결과를 출력합니다.
print(test_again.shape, same)

## 마무리 확인

- [ ] 오늘 결과가 **train에서 배운 기준으로 36열 표를 만들고 전처리기를 matchmaking_preprocessor.joblib로 저장합니다.**와 같은가?
- [ ] 실행 순서를 자기 말로 설명할 수 있는가?
- [ ] 오류가 났을 때 마지막 줄과 직전 중간 결과를 확인했는가?
